# STREAMSENSE — export_onnx.ipynb

Export trained PyTorch model to ONNX FP32 and validate against golden vectors.

Place at: `C:\STREAMSENSE\export_onnx.ipynb`

Kernel: `streamsense-env-win`

In [1]:
# Cell 1 - Install dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'onnx', 'onnxruntime', '--quiet'])
print('Dependencies ready.')

Dependencies ready.


In [2]:
# Cell 2 - Imports and paths
import sys
import json
import numpy as np
import torch
import onnx
import onnxruntime as ort
from pathlib import Path

ROOT = Path(r'C:\STREAMSENSE')
sys.path.insert(0, str(ROOT / 'training'))

from model import StreamSenseNet

CKPT_PATH   = ROOT / 'checkpoints' / 'best_model.pth'
ONNX_DIR    = ROOT / 'onnx_models'
GV_RAW_DIR  = ROOT / 'golden_vectors' / 'raw'
GV_NORM_DIR = ROOT / 'golden_vectors' / 'normalized'
MANIFEST    = ROOT / 'golden_vectors' / 'manifest.json'
FP32_PATH   = ONNX_DIR / 'streamsense_model_fp32.onnx'

ONNX_DIR.mkdir(exist_ok=True)

print('=== Path Check ===')
for p, name in [
    (CKPT_PATH,  'best_model.pth'),
    (GV_RAW_DIR, 'golden_vectors/raw'),
    (MANIFEST,   'manifest.json'),
]:
    print(f'  [{"OK" if p.exists() else "MISSING"}] {name}')

print(f'\nONNX output -> {FP32_PATH}')

=== Path Check ===
  [OK] best_model.pth
  [OK] golden_vectors/raw
  [OK] manifest.json

ONNX output -> C:\STREAMSENSE\onnx_models\streamsense_model_fp32.onnx


In [3]:
# Cell 3 - Load model
print('Loading checkpoint...')
ckpt  = torch.load(CKPT_PATH, map_location='cpu')
model = StreamSenseNet(num_classes=10)
model.load_state_dict(ckpt['model_state'])
model.eval()

print(f'  Epoch     : {ckpt["epoch"]}')
print(f'  Val acc   : {ckpt["val_accuracy"]:.2f}%')
print(f'  Eval mode : {not model.training}')

dummy = torch.zeros(1, 1, 64, 97)
with torch.no_grad():
    out = model(dummy)
print(f'  Test pass : {tuple(dummy.shape)} -> {tuple(out.shape)} OK')

Loading checkpoint...
  Epoch     : 26
  Val acc   : 96.11%
  Eval mode : True


C:\Users\shree324\AppData\Local\Temp\ipykernel_28488\1972852547.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt  = torch.load(CKPT_PATH, map_location='cpu')


  Test pass : (1, 1, 64, 97) -> (1, 10) OK


In [4]:
# Cell 4 - Export to ONNX FP32
print('Exporting to ONNX...')

dummy_input = torch.zeros(1, 1, 64, 97)

torch.onnx.export(
    model,
    dummy_input,
    str(FP32_PATH),
    opset_version       = 17,
    input_names         = ['input'],
    output_names        = ['logits'],
    dynamic_axes        = {
        'input'  : {0: 'batch'},
        'logits' : {0: 'batch'},
    },
    do_constant_folding = True,
    verbose             = False,
)

size_mb = FP32_PATH.stat().st_size / 1e6
print(f'  Exported  : {FP32_PATH.name}')
print(f'  File size : {size_mb:.2f} MB')

Exporting to ONNX...
  Exported  : streamsense_model_fp32.onnx
  File size : 1.19 MB


In [5]:
# Cell 5 - Validate ONNX model structure
print('Running onnx.checker...')

onnx_model = onnx.load(str(FP32_PATH))
onnx.checker.check_model(onnx_model)

print('  Graph check : PASSED')
print(f'  ONNX opset  : {onnx_model.opset_import[0].version}')

for inp in onnx_model.graph.input:
    shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f'  Input       : {inp.name} {shape}')

for out in onnx_model.graph.output:
    shape = [d.dim_value for d in out.type.tensor_type.shape.dim]
    print(f'  Output      : {out.name} {shape}')

Running onnx.checker...
  Graph check : PASSED
  ONNX opset  : 17
  Input       : input [0, 1, 64, 97]
  Output      : logits [0, 10]


In [6]:
# Cell 6 - Load manifest and golden vectors
with open(MANIFEST) as f:
    manifest = json.load(f)

tolerance = float(manifest['tolerance_max_abs_error'])
vectors   = manifest['vectors']

print(f'Manifest loaded.')
print(f'  Tolerance : {tolerance}')
print(f'  Vectors   : {len(vectors)}')

def load_bin(path, shape):
    arr = np.fromfile(str(path), dtype='<f4')
    return arr.reshape(shape)

print('\nLoading golden vectors...')
gv_data = []
for i in range(10):
    v    = vectors[str(i)]
    raw  = load_bin(GV_RAW_DIR  / v['raw_bin'],  tuple(v['raw_shape']))
    norm = load_bin(GV_NORM_DIR / v['norm_bin'], tuple(v['mel_shape']))
    gv_data.append({'label': v['label'], 'raw': raw, 'norm': norm})
    print(f'  GV_{i:02d}_{v["label"]}')

Manifest loaded.
  Tolerance : 0.0001
  Vectors   : 10

Loading golden vectors...
  GV_00_yes
  GV_01_no
  GV_02_up
  GV_03_down
  GV_04_left
  GV_05_right
  GV_06_on
  GV_07_off
  GV_08_stop
  GV_09_go


In [7]:
# Cell 7 - PyTorch vs ONNX Runtime comparison on all 10 golden vectors
from mel_pipeline import preprocess

sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
ort_session  = ort.InferenceSession(
    str(FP32_PATH),
    sess_options,
    providers=['CPUExecutionProvider']
)

print('=' * 60)
print('ONNX Validation - PyTorch vs ONNX Runtime')
print('=' * 60)
print(f'Tolerance : {tolerance}')
print()

passed = 0
failed = 0

for i, gv in enumerate(gv_data):
    tensor     = preprocess(gv['raw'])              # [1, 1, 64, 97]
    x          = tensor.squeeze(0).unsqueeze(0)     # [1, 1, 64, 97]

    # PyTorch
    with torch.no_grad():
        pt_logits = model(x).numpy()                # [1, 10]

    # ONNX Runtime
    ort_logits  = ort_session.run(['logits'], {'input': x.numpy()})[0]

    max_err    = float(np.max(np.abs(pt_logits - ort_logits)))
    pt_pred    = int(np.argmax(pt_logits))
    ort_pred   = int(np.argmax(ort_logits))
    top1_match = pt_pred == ort_pred
    ok         = max_err <= tolerance and top1_match

    if ok:
        passed += 1
    else:
        failed += 1

    status = 'PASS' if ok else 'FAIL'
    print(
        f'  [{status}] GV_{i:02d}_{gv["label"]:<6}  '
        f'max_err={max_err:.2e}  '
        f'PT_pred={pt_pred}  ORT_pred={ort_pred}  '
        f'top1={top1_match}'
    )

print()
print('=' * 60)
print(f'Passed: {passed}/10   Failed: {failed}/10')
if failed == 0:
    print('[PASS] streamsense_model_fp32.onnx validated and ready.')
    print('Next step: quantize_ptq.ipynb')
else:
    print('[FAIL] Export validation failed. Do not proceed to quantize.')
print('=' * 60)

ONNX Validation - PyTorch vs ONNX Runtime
Tolerance : 0.0001

  [PASS] GV_00_yes     max_err=2.86e-06  PT_pred=0  ORT_pred=0  top1=True
  [PASS] GV_01_no      max_err=2.86e-06  PT_pred=1  ORT_pred=1  top1=True
  [PASS] GV_02_up      max_err=1.91e-06  PT_pred=2  ORT_pred=2  top1=True
  [PASS] GV_03_down    max_err=2.86e-06  PT_pred=3  ORT_pred=3  top1=True
  [PASS] GV_04_left    max_err=3.81e-06  PT_pred=4  ORT_pred=4  top1=True
  [PASS] GV_05_right   max_err=1.43e-06  PT_pred=5  ORT_pred=5  top1=True
  [PASS] GV_06_on      max_err=3.81e-06  PT_pred=6  ORT_pred=6  top1=True
  [PASS] GV_07_off     max_err=9.54e-07  PT_pred=7  ORT_pred=7  top1=True
  [PASS] GV_08_stop    max_err=1.91e-06  PT_pred=8  ORT_pred=8  top1=True
  [PASS] GV_09_go      max_err=1.91e-06  PT_pred=9  ORT_pred=9  top1=True

Passed: 10/10   Failed: 0/10
[PASS] streamsense_model_fp32.onnx validated and ready.
Next step: quantize_ptq.ipynb


In [8]:
# Cell 8 - Summary
print('=== Export Summary ===')
print(f'  Model      : StreamSenseNet')
print(f'  Parameters : 295,786')
print(f'  Checkpoint : epoch {ckpt["epoch"]}, val_acc={ckpt["val_accuracy"]:.2f}%')
print(f'  ONNX file  : {FP32_PATH}')
print(f'  File size  : {FP32_PATH.stat().st_size / 1e6:.2f} MB')
print(f'  Opset      : 17')
print(f'  Input      : [batch, 1, 64, 97] float32')
print(f'  Output     : [batch, 10] float32 logits')
print(f'  Validation : {passed}/10 golden vectors PASS')

=== Export Summary ===
  Model      : StreamSenseNet
  Parameters : 295,786
  Checkpoint : epoch 26, val_acc=96.11%
  ONNX file  : C:\STREAMSENSE\onnx_models\streamsense_model_fp32.onnx
  File size  : 1.19 MB
  Opset      : 17
  Input      : [batch, 1, 64, 97] float32
  Output     : [batch, 10] float32 logits
  Validation : 10/10 golden vectors PASS
